# Problem Formulation and Limitation Audit

This notebook critically evaluates the current formulation of the learning-based algorithm selection problem.

The analysis focuses on:

1. whether the current solver portfolio creates a meaningful selection problem;
2. the sensitivity of labels to the runtime trade-off parameter;
3. possible shortcut relationships between instance size and solver labels;
4. the stability and practical interpretation of the current target;
5. limitations that should be addressed before further model development.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
current_dir = Path.cwd().resolve()

if (current_dir / "results").exists():
    project_root = current_dir
elif (current_dir.parent / "results").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the results directory."
    )

solver_path = project_root / "results" / "solver_results_v2.csv"
ml_path = project_root / "results" / "algorithm_selection_dataset.csv"

solver_df = pd.read_csv(solver_path)
ml_df = pd.read_csv(ml_path)

print("Project root:", project_root)
print("Solver results shape:", solver_df.shape)
print("ML dataset shape:", ml_df.shape)

## 1. Data Integrity Check

Before analysing the formulation, the consistency of the solver results
and machine-learning dataset is verified.

In [ ]:
integrity_summary = pd.Series({
    "solver_rows": len(solver_df),
    "ml_rows": len(ml_df),
    "solver_missing_values": solver_df.isna().sum().sum(),
    "ml_missing_values": ml_df.isna().sum().sum(),
    "duplicate_solver_ids": solver_df["id"].duplicated().sum(),
    "expected_sequential_ids": (
        solver_df["id"].to_numpy()
        == np.arange(len(solver_df))
    ).all()
})

integrity_summary

In [ ]:
alignment_check = pd.DataFrame({
    "solver_n_items": solver_df["n_items"],
    "ml_n_items": ml_df["n_items"],
    "solver_correlation": solver_df["correlation"],
    "ml_correlation": ml_df["correlation"]
})

print(
    "n_items aligned:",
    (alignment_check["solver_n_items"]
     == alignment_check["ml_n_items"]).all()
)

print(
    "correlation aligned:",
    (alignment_check["solver_correlation"]
     == alignment_check["ml_correlation"]).all()
)

## 2. Solution-Quality Difference

Dynamic programming always provides an optimal solution in the current
setting. However, the practical value of algorithm selection depends on
whether the greedy heuristic produces meaningfully different solutions.

In [ ]:
gap_summary = solver_df["gap"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
)

gap_summary

In [ ]:
quality_summary = pd.Series({
    "mean_gap_percent": solver_df["gap"].mean() * 100,
    "median_gap_percent": solver_df["gap"].median() * 100,
    "maximum_gap_percent": solver_df["gap"].max() * 100,
    "greedy_exact_count": (solver_df["gap"] == 0).sum(),
    "greedy_exact_rate_percent": (
        (solver_df["gap"] == 0).mean() * 100
    ),
    "gap_above_1_percent_count": (
        solver_df["gap"] > 0.01
    ).sum()
})

quality_summary

In [ ]:
plt.figure(figsize=(8, 4))

plt.hist(
    solver_df["gap"] * 100,
    bins=30,
    edgecolor="black"
)

plt.xlabel("Greedy Optimality Gap (%)")
plt.ylabel("Number of Instances")
plt.title("Distribution of Greedy Optimality Gaps")
plt.show()

### Preliminary Observation

The greedy heuristic is either optimal or very close to the dynamic
programming solution for a large proportion of the current instances.

This reduces the practical consequence of selecting the wrong solver.
A classifier may predict the preferred algorithm accurately without
producing a meaningful improvement in optimization performance.

Therefore, classification accuracy alone is insufficient for evaluating
the usefulness of the selector.

## 3. Sensitivity of the Cost-Aware Target

The current target is defined using:

$$
S_a = P_a - \lambda T_a
$$

where $$P_a\ and\ T_a\ $$ represent the profit and runtime of algorithm $$a\ $$ The following analysis recomputes the label distribution directly
from the solver results.

In [ ]:
def generate_cost_aware_labels(df, lambda_runtime):
    greedy_score = (
        df["greedy_profit"]
        - lambda_runtime * df["greedy_time"]
    )

    dp_score = (
        df["dp_profit"]
        - lambda_runtime * df["dp_time"]
    )

    labels = np.where(
        greedy_score > dp_score,
        "greedy",
        "dp"
    )

    return pd.Series(
        labels,
        index=df.index,
        name="best_algorithm"
    )

In [ ]:
lambda_values = [
    0,
    10,
    25,
    50,
    100,
    250,
    500,
    1000,
    2000,
    5000
]

lambda_records = []

for lambda_runtime in lambda_values:
    labels = generate_cost_aware_labels(
        solver_df,
        lambda_runtime
    )

    counts = labels.value_counts()

    lambda_records.append({
        "lambda": lambda_runtime,
        "greedy": counts.get("greedy", 0),
        "dp": counts.get("dp", 0),
        "greedy_rate": (
            counts.get("greedy", 0) / len(labels)
        )
    })

lambda_df = pd.DataFrame(lambda_records)

lambda_df

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    lambda_df["lambda"],
    lambda_df["greedy_rate"],
    marker="o"
)

plt.axhline(
    0.5,
    color="grey",
    linestyle="--",
    linewidth=1
)

plt.xlabel("Runtime Trade-off Parameter (λ)")
plt.ylabel("Proportion Labelled as Greedy")
plt.title("Label Sensitivity to Runtime Trade-off")
plt.grid(alpha=0.3)
plt.show()

### Preliminary Observation

The class distribution changes substantially with the value of λ.
Therefore, the machine-learning target is not an inherent property of
an instance; it also reflects a user-defined preference between solution
quality and runtime.

A suitable research formulation must either justify λ using an
application-specific cost model or replace the weighted score with a
more interpretable decision criterion, such as a fixed runtime budget,
quality threshold, or selection regret.

## 4. Potential Shortcut Relationships

A useful algorithm selector should learn meaningful interactions between
problem characteristics and solver performance. If labels are largely
determined by problem size, a classifier may learn a simple threshold
rather than general algorithmic behaviour.

In [ ]:
solver_audit = solver_df.copy()

solver_audit["label_lambda_100"] = (
    generate_cost_aware_labels(
        solver_audit,
        lambda_runtime=100
    )
)

size_label_table = pd.crosstab(
    solver_audit["n_items"],
    solver_audit["label_lambda_100"],
    margins=True
)

size_label_table

In [ ]:
size_label_rate = pd.crosstab(
    solver_audit["n_items"],
    solver_audit["label_lambda_100"],
    normalize="index"
)

size_label_rate

In [ ]:
correlation_label_table = pd.crosstab(
    solver_audit["correlation"],
    solver_audit["label_lambda_100"],
    margins=True
)

correlation_label_table

In [ ]:
size_label_rate.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4)
)

plt.ylabel("Class Proportion")
plt.xlabel("Number of Items")
plt.title("Cost-Aware Labels by Problem Size")
plt.legend(title="Selected Algorithm")
plt.show()

### Preliminary Observation

At λ=100, all current instances with 200 items are labelled as greedy.
This indicates that problem size may act as a strong shortcut feature.

The existing machine-learning accuracy may therefore partly reflect the
ability to learn runtime thresholds associated with n_items rather than
a deeper relationship between instance structure and solver suitability.

In [ ]:
lambda_runtime = 100

greedy_score = (
    solver_df["greedy_profit"]
    - lambda_runtime * solver_df["greedy_time"]
)

dp_score = (
    solver_df["dp_profit"]
    - lambda_runtime * solver_df["dp_time"]
)

score_margin = (greedy_score - dp_score).abs()

score_margin_summary = score_margin.describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
)

score_margin_summary

In [ ]:
margin_counts = pd.Series({
    "margin_below_1": (score_margin < 1).sum(),
    "margin_below_5": (score_margin < 5).sum(),
    "margin_below_10": (score_margin < 10).sum()
})

margin_counts

### Preliminary Observation

A substantial number of instances have small cost-aware score margins.
Since runtimes are currently measured only once, measurement noise could
change the preferred label for near-tie instances.

Runtime stability should therefore be evaluated using repeated
measurements, warm-up runs, and robust statistics such as the median
runtime.